<h1 align="center">Hoja de Trabajo 2</h1>

## Información

**Integrantes:**

| Name              | Institution ID | GitHub User |
| ----------------- | -------------- | ----------- |
| Josué Say         | 22801          | JosueSay    |
| Carlos Valladares | 221164         | vgcarlol    |

- [Repositorio](https://github.com/JosueSay/intro-to-computer-vision/tree/main/worksheets/ws2)

## Preparación de entorno

In [ ]:
# %pip install -r requirements.txt
# jupyter nbconvert ws2.ipynb --to html

## Task 1 - Análisis

Considere que usted está diseñando el sistema de visión para un robot de almacén que debe moverse entre estanterías para recoger productos. El robot tiene dos cámaras frontales:

### Inciso 1

Durante una prueba, el robot gira sobre su propio eje para escanear el entorno. El ingeniero junior a tu cargo sugiere usar Homografías para medir la distancia a los objetos mientras el robot gira. ¿Es este un enfoque correcto? Justifique su respuesta utilizando los conceptos de C1, C2 y Paralaje.

**Respuesta:**

No, porque al ser estanterías se deberá medir distancia mientras el robot solo gira.

- Si el robot gira sobre su eje, el centro de la cámara no se mueve: $C_1 = C_2$ (no hay traslación).
- Cuando $C_1 = C_2$ entonces el paralaje es $\approx$ 0.
- Sin paralaje no hay disparidad útil, y por lo tanto se puede triangular profundidad.
- La homografía sirve para relacionar dos vistas bajo rotación o cuando la escena puede aproximarse como plana; es útil para alinear/“stitching”, no para medir distancia 3D real en estanterías con objetos a diferentes profundidades.

> Con giro puro se puede usar homografías para registrar imágenes, pero no para estimar distancias a objetos en 3D.

### Inciso 2

Si el robot comienza a avanzar (traslación) y detectas que la disparidad (d) de una caja aumenta repentinamente entre el frame t y el frame t+1, ¿qué puedes inferir sobre la distancia (Z) entre el robot y la caja? ¿Qué riesgo industrial implicaría un error en el cálculo de esta disparidad?

**Respuesta:**

Por la relación $Z = \frac{fB}{d}$, si $d \uparrow$ ⇒ $Z \downarrow$, es decir la caja está más cerca del robot.

**Riesgo industrial si $d$ se calcula mal:**

- Un error en disparidad produce un error directo en profundidad ($Z$), causando:

  - Choque con las estanterias/cajas.
  - El robot fallará cuando intente agarrar la caja (está mas cerca/lejos de donde está realmente).
    - Esto a su vez puede llegar a dañar el producto de las estanterías aumento pérdidas monetarias.



## Task 2 – Ingeniería de Dimensiones

Como director de proyectos, debe asegurar que los modelos de IA quepan en la memoria de los dispositivos (Edge Computing). Por ello, tiene una imagen de entrada de alta resolución proveniente de una cámara industrial de 1280 x 720 píxeles. Se aplica una capa convolucional con los siguientes hiperparámetros:

- Tamaño del Filtro (F): 5×5  
- Padding (P): 2  
- Stride (S): 2  

Considerando esto, respondan:

### Inciso 1

Utilizando la fórmula vista en clase, calcule las dimensiones (Wout,Hout) del Mapa de Características resultante. Muestra el procedimiento.

**Respuesta:**

Fórmula:

$$
O = \left\lfloor \frac{W - F + 2P}{S} \right\rfloor + 1
$$

Datos:

- $W = 1280$
- $H = 720$
- $F = 5$
- $P = 2$
- $S = 2$

**Cálculo en ancho**

$$
W_{out} = \left\lfloor \frac{1280 - 5 + 2(2)}{2} \right\rfloor + 1
$$

$$
W_{out} = \left\lfloor \frac{1280 - 5 + 4}{2} \right\rfloor + 1
$$

$$
W_{out} = \left\lfloor \frac{1279}{2} \right\rfloor + 1
$$

$$
W_{out} = \left\lfloor 639.5 \right\rfloor + 1
$$

$$
W_{out} = 639 + 1 = 640
$$

**Cálculo en alto**

$$
H_{out} = \left\lfloor \frac{720 - 5 + 4}{2} \right\rfloor + 1
$$

$$
H_{out} = \left\lfloor \frac{719}{2} \right\rfloor + 1
$$

$$
H_{out} = \left\lfloor 359.5 \right\rfloor + 1
$$

$$
H_{out} = 359 + 1 = 360
$$


**Resultado**

$$
( W_{out}, H_{out} ) = (640, 360)
$$


### Inciso 2

¿Qué sucedería con el tamaño de la salida si decides cambiar el Padding a P=0 (Valid Padding)? ¿Cómo afectaría esto a la información de los bordes de la imagen (donde suelen estar las referencias de las paredes del almacén)?

**Respuesta:**

Fórmula:

$$
O = \left\lfloor \frac{W - F}{S} \right\rfloor + 1
$$

**Ancho**

$$
W_{out} = \left\lfloor \frac{1280 - 5}{2} \right\rfloor + 1
$$

$$
W_{out} = \left\lfloor \frac{1275}{2} \right\rfloor + 1
$$

$$
W_{out} = 637 + 1 = 638
$$

**Alto**

$$
H_{out} = \left\lfloor \frac{720 - 5}{2} \right\rfloor + 1
$$

$$
H_{out} = \left\lfloor \frac{715}{2} \right\rfloor + 1
$$

$$
H_{out} = 357 + 1 = 358
$$

**Nuevo tamaño**

$$
(638, 358)
$$

Con $P=0$ (valid padding):

- La convolución hace que la imagen se reduzca naturalmente, porque el kernel no logra meterlos en los bordes.
- El centro del filtro no puede llegar a las esquinas, por lo que esos píxeles se pierden.
- Dado que no debemos hacer de menos la información de los límites, porque también contiene relaciones espaciales.

En el contexto del robot de almacén:

- Las paredes y estanterías podrían aparecer en los extremos del frame.
- Si eliminamos esos píxeles, estamos perdiendo referencias espaciales afectando en el funcionamiento del robot porque no interpreta bien la percepción de límite.

Aunque reduce dimensiones y puede ayudar en memoria, también implica pérdida de información en bordes llegando a perder los detalles periféricos que son importantes para la geometría y orientación del robot.

## Task 3 – Criterio de Diseño

En la industria, el balance entre precisión y velocidad es clave. Analice los siguientes escenarios:

### Inciso 1

Usted está desarrollando un sistema de detección de grietas microscópicas en motores de avión. ¿Qué combinación de Stride y Pooling recomendaría para no perder detalles críticos en las primeras capas de la red? Justifique técnicamente.

**Respuesta:**



### Inciso 1

Un cliente le pide que el sistema funcione en un procesador muy limitado (como una cámara inteligente con poca RAM). Explique cómo podrías utilizar el Stride y el Max Pooling estratégicamente para reducir la carga computacional sin eliminar las características más fuertes (activaciones) del Mapa de Características

**Respuesta:**



## Task 4 – Implementación Práctica

Con esta parte se busca que puedan comprender la mecánica interna de la operación convolucional sin depender de librerías de alto nivel para la lógica central.Por ello realice lo siguiente:

- Utilizando un lenguaje de programación (Python) y librerías básicas para manejo de matrices (como NumPy), implementa una función llamada `manual_convolution(image, kernel, stride, padding)`.

- **Requisitos de la implementación:**
  - La función debe recibir una matriz 2D (imagen en escala de grises) y un filtro (kernel) de tamaño N×N.
  - Debe aplicar primero el Zero-padding a la imagen de entrada según el valor de P.
  - Debe recorrer la imagen aplicando el producto punto (suma de productos elementales) respetando el Stride indicado.
  - La función debe retornar la matriz resultante (Feature Map).

- **Prueba de validación:** Defina un filtro de detección de bordes verticales (filtro de Sobel o similar) y aplíquelo a una imagen pequeña de prueba.

- **Entrega:** Suba su Jupyter Notebook con comentarios explicando cómo el desplazamiento del kernel afecta el tamaño de la matriz de salida.